# 融合 Softmax

本教程将编写一个融合的 Softmax 算子。对于每一行都能放入 GPU SRAM 的矩阵，教程旨在使其明显快于 PyTorch 原生实现。

你将学习：

- 内核融合对受内存带宽限制的运算有什么好处。
- Triton 中的归约运算。
---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html)，版本为 2026-10-16 下载的官网 main 文档。

中文练习版：保留说明与公式；原有代码单元及说明中的独立代码块均保留为空白代码框；移除代码、运行输出、测量结果与图表。空白框供自行填写，未运行教程。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

## 动机

为逐元素加法编写自定义 GPU 内核很适合入门，但实际应用价值有限。接下来考虑一个经过数值稳定处理的简单 Softmax 运算。

对于输入 $x \in \mathbb{R}^{M \times N}$，朴素的 PyTorch 实现需要从 DRAM 读取 $5MN+2M$ 个元素，并写回 $3MN+2M$ 个元素。

这些访存开销很浪费。理想的融合内核只读取一次输入 X，并在芯片内部完成所需计算。这样只需读取 $MN$ 个元素、写回 $MN$ 个元素，理论加速比约为 4 倍，即 $(8MN+4M)/(2MN)$。

原文将这里的 $MN$ 写为“字节”；按前后文的访存计数，应理解为元素数量，实际字节数还要乘以每个元素的大小。

TorchScript 的脚本编译机制试图自动完成这种内核融合，但原教程指出，其效果仍可能不理想。

## 计算内核

每个程序实例以程序实例总数为步长，处理输入矩阵 X 的一组行：加载数据、完成归一化，然后将结果写入输出 Y。

这里需要注意，Triton 的块元素数量要求是 2 的幂。为了支持任意输入形状，需要在内部对各行进行填充，并恰当地保护内存访问。

可以创建一个辅助函数，为给定的输入张量启动内核，并传入所需参数和元参数。

## 单元测试

使用行数和列数都不规则的矩阵测试内核，以检验填充机制是否正确。

检查自定义内核的结果是否与参考实现一致。

## 性能基准测试

固定输入矩阵为 4096 行，改变列数来测试算子的性能，并与 PyTorch 原生 Softmax 以及前面定义的朴素实现进行比较。

原教程在此讨论了融合实现相对于 Torch JIT 和 PyTorch 原生实现的表现，并指出自定义实现便于阅读、理解和维护。本笔记已移除测量数值及图表。

需要注意，PyTorch 的 Softmax 是更通用的操作，支持任意形状的张量。